# IndoBERT Clickbait Detection - Complete Pipeline

This notebook runs the complete training and evaluation pipeline for IndoBERT-based clickbait detection with robustness evaluation.

**Requirements:**
- GPU Runtime (T4 for testing, A100 for full training)
- 5 CSV files with columns: `text`, `label`, `domain`
- ~(TBA) hours for complete execution

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository or upload code files
# Option 1: Clone from GitHub
!git clone https://github.com/gredss/domain-shift-trustworthy-ai.git
%cd domain-shift-trustworthy-ai

# Option 2: Upload files manually
from google.colab import files
import os

print("Upload all Python files (.py) from your project")
uploaded = files.upload()

# List uploaded files
!ls -lh *.py

In [67]:
# Install dependencies
!pip install -q torch transformers pandas numpy scikit-learn scipy streamlit plotly tqdm

## 2. Data Preparation

In [68]:
# Create necessary directories
!mkdir -p dataset
!mkdir -p /content/drive/MyDrive/thesis/checkpoints
!mkdir -p /content/drive/MyDrive/thesis/results

In [ ]:
# Upload your 5 CSV files
print("Upload your 5 domain CSV files (Technology, Politics, Health, Sport, Education)")
uploaded_data = files.upload()

# Move to dataset directory
import shutil
for filename in uploaded_data.keys():
    shutil.move(filename, f'dataset/{filename}')
    
# Verify uploaded files
!ls -lh dataset/

In [ ]:
# Quick data verification from the uploaded CSV files
import pandas as pd
import glob

csv_files = glob.glob('dataset/*.csv')
print(f"Found {len(csv_files)} CSV files\n")

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    print(f"{csv_file}:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Label distribution: {df['Label'].value_counts().to_dict()}")
    print()

In [ ]:
# # Quick data verification from the existing dataset/data folder CSV files
# import pandas as pd
# import glob

# csv_files = glob.glob('dataset/data/*.csv')
# print(f"Found {len(csv_files)} CSV files\n")

# for csv_file in csv_files:
#     df = pd.read_csv(csv_file, sep=';')
#     print(f"{csv_file}:")
#     print(f"  Rows: {len(df)}")
#     print(f"  Columns: {list(df.columns)}")
#     print(f"  Label distribution: {df['Label'].value_counts().to_dict()}")
#     print()

## 3. Model Training

Train IndoBERT models. Start with base model for testing, then scale to all models.

In [ ]:
# Train base model (recommended for initial testing)
!python src/train_pipeline.py \
    --model base \
    --dataset-dir dataset/data \
    --checkpoint-dir /content/drive/MyDrive/thesis/checkpoints \
    --output-dir /content/drive/MyDrive/thesis/output \
    --device cuda \
    --seed 42

In [ ]:
# Train all models (base, large, lite) - Use A100 GPU for this
# Uncomment to run:
# !python src/train_pipeline.py \
#     --model all \
#     --dataset-dir dataset \
#     --checkpoint-dir /content/drive/MyDrive/thesis/checkpoints \
#     --output-dir /content/drive/MyDrive/thesis/output \
#     --device cuda \
#     --seed 42

In [ ]:
# Optional: Train with hyperparameter grid search
# !python src/train_pipeline.py \
#     --model base \
#     --dataset-dir dataset \
#     --checkpoint-dir /content/drive/MyDrive/thesis/checkpoints \
#     --output-dir /content/drive/MyDrive/thesis/output \
#     --device cuda \
#     --grid-search \
#     --seed 42

## 4. Model Evaluation

Run comprehensive evaluation including in-domain, cross-domain, and perturbation testing.

In [ ]:
# Evaluate base model with full perturbation testing
!python src/evaluate_pipeline.py \
    --model base \
    --checkpoint-dir /content/drive/MyDrive/thesis/checkpoints \
    --dataset-dir dataset/data \
    --output-dir /content/drive/MyDrive/thesis/results \
    --device cuda \
    --seed 42

In [ ]:
# Quick evaluation (skip perturbation testing for faster results)
# !python src/evaluate_pipeline.py \
#     --model base \
#     --checkpoint-dir /content/drive/MyDrive/thesis/checkpoints \
#     --dataset-dir dataset \
#     --output-dir /content/drive/MyDrive/thesis/results \
#     --device cuda \
#     --skip-perturbation \
#     --seed 42

In [ ]:
# Evaluate all models (if you trained all)
# !python src/evaluate_pipeline.py \
#     --model all \
#     --checkpoint-dir /content/drive/MyDrive/thesis/checkpoints \
#     --dataset-dir dataset \
#     --output-dir /content/drive/MyDrive/thesis/results \
#     --device cuda \
#     --seed 42

## 5. View Results

In [ ]:
# View training summary
import json

with open('/content/drive/MyDrive/thesis/output/training_summary.json', 'r') as f:
    training_summary = json.load(f)

print("Training Summary:")
print(f"Total samples: {training_summary['total_samples']}")
print(f"Domains: {', '.join(training_summary['domains'])}")
print(f"\nBest F1 Scores:")
for model, results in training_summary['training_results'].items():
    print(f"  {model}: {results['best_f1']:.4f}")

In [ ]:
# View evaluation summary
with open('/content/drive/MyDrive/thesis/results/evaluation_summary.json', 'r') as f:
    eval_summary = json.load(f)

print("Evaluation Summary:")
for model, metrics in eval_summary['model_summaries'].items():
    print(f"\n{model.upper()}:")
    print(f"  Avg In-Domain F1: {metrics['avg_in_domain_f1']:.4f}")
    print(f"  Avg Cross-Domain F1: {metrics['avg_cross_domain_f1']:.4f}")
    if 'avg_perturbation_f1' in metrics:
        print(f"  Avg Perturbation F1: {metrics['avg_perturbation_f1']:.4f}")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import seaborn as sns

# Plot F1 scores comparison
models = list(eval_summary['model_summaries'].keys())
in_domain_f1 = [eval_summary['model_summaries'][m]['avg_in_domain_f1'] for m in models]
cross_domain_f1 = [eval_summary['model_summaries'][m]['avg_cross_domain_f1'] for m in models]

x = range(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([i - width/2 for i in x], in_domain_f1, width, label='In-Domain', alpha=0.8)
ax.bar([i + width/2 for i in x], cross_domain_f1, width, label='Cross-Domain', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('F1 Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/thesis/results/performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Launch Dashboard (Optional)

Launch the interactive Streamlit dashboard for visualization and testing.

In [ ]:
# Install localtunnel for public URL
!npm install -g localtunnel

In [ ]:
# Launch dashboard in background
import subprocess
import threading
import time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'dashboard_app.py', '--server.port', '8501'])

# Start Streamlit in background
thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

time.sleep(5)
print("Streamlit is starting...")

In [ ]:
# Create public URL with localtunnel
!lt --port 8501

## 7. Download Results

In [ ]:
# Create zip file of all results
!zip -r results.zip /content/drive/MyDrive/thesis/results/
!zip -r checkpoints.zip /content/drive/MyDrive/thesis/checkpoints/

print("Results and checkpoints zipped successfully!")
print("Files are saved in Google Drive and can be downloaded from there.")